# Phase 6 — Segmentation Feasibility Audit

This notebook is a read-only feasibility audit for candidate segmentation annotations. It does not train a U-Net and does not assume that directories named `segmented`, `mask`, or `lesion` contain valid disease-lesion ground truth.

Objective: determine whether the currently available PlantVillage data provides defensible disease-lesion segmentation annotations, or whether the feasibility of later U-Net training remains unestablished.

## 1) Project and data-path verification

In [ ]:
from pathlib import Path
import os

project_root = Path.cwd()
if project_root.name != 'tomato_disease_framework':
    matches = list(Path.cwd().glob('**/tomato_disease_framework'))
    if matches:
        project_root = matches[0]

os.chdir(project_root)
print(f'Project root: {project_root}')

TDF_DATASET_ROOT = '/content/drive/MyDrive/UROP_Plant_Disease/datasets/PlantVillage/plantvillage dataset/color'
os.environ['TDF_DATASET_ROOT'] = TDF_DATASET_ROOT
dataset_root = Path(os.environ['TDF_DATASET_ROOT'])
print('Dataset root:', dataset_root)
print('Dataset exists:', dataset_root.exists())
if not dataset_root.exists():
    print('The dataset is not mounted in the current runtime. The audit will still produce a conservative report.')


## 2) Environment verification

In [ ]:
import sys
import matplotlib
import numpy as np
from PIL import Image

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Pillow:', Image.__version__)
print('Matplotlib:', matplotlib.__version__)
print('Environment verification OK')


## 3) Run the audit script

In [ ]:
!python scripts/audit_segmentation_annotations.py

## 4) Candidate inventory statistics

In [ ]:
import pandas as pd
import yaml

inventory_path = project_root / 'results' / 'phase6' / 'segmentation_candidate_inventory.csv'
report_path = project_root / 'results' / 'phase6' / 'segmentation_feasibility_report.yaml'

inventory = pd.read_csv(inventory_path) if inventory_path.exists() else pd.DataFrame()
with open(report_path, 'r', encoding='utf-8') as handle:
    report = yaml.safe_load(handle) if report_path.exists() else {}

print('Inventory rows:', len(inventory))
display(inventory.head(20) if not inventory.empty else pd.DataFrame({'status': ['No candidate inventory available']}))
print('Audit summary:')
for key, value in (report.get('audit_summary', {}) or {}).items():
    print(f'  {key}: {value}')
print('Evidence-based questions:')
for key, value in (report.get('questions', {}) or {}).items():
    print(f'  {key}: {value}')


## 5) Source-to-candidate pairing analysis

In [ ]:
pairing = report.get('pairing', {}) if report else {}
print('Total color images:', pairing.get('total_color_images', 'N/A'))
print('Total candidate annotation files:', pairing.get('total_candidate_annotation_files', 'N/A'))
print('Matched pairs:', pairing.get('matched_pairs', 'N/A'))
print('Unmatched candidate files:', pairing.get('unmatched_candidate_files', 'N/A'))
print('Unmatched source images:', pairing.get('unmatched_source_images', 'N/A'))
print('Duplicate stems:', pairing.get('duplicate_stems', []))
print('Obvious filename mismatches:')
for item in pairing.get('obvious_filename_mismatches', [])[:10]:
    print('  -', item)


## 6) Image and mask dimensions / mode analysis

The audit reports raw image dimensions and modes, but the key scientific point is: a binary-looking image or foreground/background segmentation does not automatically prove disease-lesion annotation ground truth.

In [ ]:
if not inventory.empty:
    cols = ['file_count', 'widths', 'heights', 'modes', 'bands', 'binary_like_count']
    display(inventory[cols].head(20))
else:
    print('No candidate inventory rows available; no candidate collection was discovered.')


## 7) Pixel-value / binary-like analysis

In [ ]:
if not inventory.empty:
    display(inventory[['binary_like_count', 'sample_unique_values', 'sample_foreground_background']].head(20))
else:
    print('No sampled binary-like statistics were produced.')


## 8) Class coverage analysis where possible

In [ ]:
if not inventory.empty:
    class_coverage = inventory[['path', 'class_counts']].copy()
    display(class_coverage.head())
else:
    print('Class coverage could not be determined from the current file inventory.')


## 9) Representative visual inspection of candidate files

A leaf/background segmentation image is not the same thing as a verified disease-lesion mask. Foreground/background separation can be a valid segmentation task, but without explicit lesion annotation it does not constitute disease-region ground truth.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

pairs = report.get('pairing', {}).get('matched_pair_examples', [])

if pairs:
    for pair in pairs[:3]:
        source_path = Path(pair['source'])
        candidate_path = Path(pair['candidate'])
        if source_path.exists() and candidate_path.exists():
            fig, axes = plt.subplots(1, 2, figsize=(8, 4))
            axes[0].imshow(Image.open(source_path))
            axes[0].set_title('Source image')
            axes[1].imshow(Image.open(candidate_path), cmap='gray')
            axes[1].set_title('Candidate segmentation image')
            for ax in axes:
                ax.axis('off')
            plt.tight_layout()
            plt.show()
        else:
            print(f'Skipping unavailable pair: {pair}')
else:
    print('No valid one-to-one source/candidate pairs were identified in the current audit. Manual review of candidate collections is still required.')


## 10) Evidence-based feasibility decision

In [ ]:
print(report.get('conclusion', 'No conclusion available.'))
print('Status answers:')
for key in ['A', 'B', 'C', 'D', 'E', 'F', 'G']:
print(f"  {key}: {report.get('questions', {}).get(key, 'NOT ESTABLISHED')}")


## 11) Final Phase 6 conclusion

- Disease-lesion annotations were not established.
- Candidate images may look like leaf/background segmentation, but this does not constitute verified lesion ground truth.
- U-Net disease-lesion training is not justified using the currently verified annotations.